# Week 2 · Day 3 — Date Functions
**Goal:** Extract, calculate, and filter on dates to answer time-based business questions.

**Dataset:** Canadian logistics — shipments (ship_date, delivery_date), drivers (hire_date)

**Schedule:**
- Block 1 · 9:00–10:30am · Concept + Drills
- Block 2 · 11:00am–12:00pm · Apply + AI Audit

**Foundation coming in:** JOINs, GROUP BY, subqueries, CTEs, window functions

---
## Business First Prompt

> *The operations manager asks: "I need to know how our freight volume has trended month by month this year, which quarter was our strongest, how long deliveries are taking on average, and whether drivers hired more recently are outperforming veterans. All of this needs to be answerable from the data we have."*

Write 3–5 sentences below before touching any data:
- What date columns exist and what questions can each one answer?
- What does "month by month" require you to extract from a date?
- How would you calculate delivery time from two date columns?

**Your answer:**

> To answer this, I would first identify the key date columns available such as ship_date, delivery_date, and hire_date, since each supports a different part of the analysis: shipping trends, delivery performance, and driver tenure. ship_date would be used to analyze freight volume trends and quarterly performance, while hire_date would allow me to segment drivers into newer versus experienced groups. “Month by month” requires extracting the year and month from the date field, typically using something like strftime('%Y-%m', ship_date) so that shipments can be grouped into consistent monthly buckets. Delivery time would be calculated by using JULIANDAY(delivery_date) - JULIANDAY(ship_date) to get the number of days between shipment and delivery.

---
## Setup — run this first

In [8]:
import sqlite3
import pandas as pd

conn = sqlite3.connect(r'C:\Users\gianc\Documents\Logistics Operations Project\logistics_w2.db')
print('Connected to logistics_w2.db')

Connected to logistics_w2.db


---
## Table Preview

In [9]:
for table in ['clients','warehouses','vehicles','drivers','routes','shipments','shipment_items']:
    print(f'\n--- {table} ---')
    df = pd.read_sql_query(f'SELECT * FROM {table} LIMIT 3', conn)
    print(f'Columns: {list(df.columns)}')
    display(df)


--- clients ---
Columns: ['client_id', 'company', 'city', 'province', 'segment']


,client_id,company,city,province,segment
0,1,Maple Retail Co,Toronto,ON,Enterprise
1,2,Pacific Goods Ltd,Vancouver,BC,Enterprise
2,3,Prairie Supply Inc,Calgary,AB,SMB



--- warehouses ---
Columns: ['warehouse_id', 'name', 'city', 'province', 'capacity_sqft']


,warehouse_id,name,city,province,capacity_sqft
0,1,Toronto Central,Toronto,ON,85000
1,2,Vancouver Hub,Vancouver,BC,72000
2,3,Calgary Depot,Calgary,AB,55000



--- vehicles ---
Columns: ['vehicle_id', 'plate', 'type', 'max_kg']


...

,route_id,origin_wh_id,dest_city,dest_province,distance_km
0,1,1,Montreal,QC,541.0
1,2,1,Ottawa,ON,450.0
2,3,1,Halifax,NS,1800.0



--- shipments ---
Columns: ['shipment_id', 'client_id', 'route_id', 'driver_id', 'vehicle_id', 'ship_date', 'delivery_date', 'status', 'freight_charge']


,shipment_id,client_id,route_id,driver_id,vehicle_id,ship_date,delivery_date,status,freight_charge
0,1001,4,4,3,2,2024-11-23,2024-11-24,Delivered,3375.74
1,1002,2,10,7,1,2024-12-12,None,Delayed,4030.98
2,1003,10,1,9,4,2024-01-16,2024-01-20,Delivered,1162.07



--- shipment_items ---
Columns: ['item_id', 'shipment_id', 'description', 'category', 'weight_kg', 'declared_value']


,item_id,shipment_id,description,category,weight_kg,declared_value
0,1,1001,Electronics,Standard,787.6,321.17
1,2,1002,Office Supplies,Standard,614.3,3832.60
2,3,1002,Clothing,Standard,97.7,2512.06


---
## Concept 1 — Extracting Parts of a Date with strftime()

In SQLite, `strftime()` is the main function for extracting date parts. It takes a format string and a date column.

| Format | What it returns | Example |
|---|---|---|
| `'%Y'` | 4-digit year | '2024' |
| `'%m'` | 2-digit month | '01' through '12' |
| `'%d'` | 2-digit day | '01' through '31' |
| `'%Y-%m'` | Year-month | '2024-03' |
| `'%Y-Q%q'` | Year-quarter (not in SQLite) | ❌ use CASE WHEN instead |

```sql
-- Extract year
strftime('%Y', ship_date)        -- '2024'

-- Extract month
strftime('%m', ship_date)        -- '03'

-- Year-month for grouping
strftime('%Y-%m', ship_date)     -- '2024-03'

-- Cast to integer for comparisons
CAST(strftime('%m', ship_date) AS INTEGER)  -- 3
```

**SQLite quirk:** SQLite stores dates as TEXT in 'YYYY-MM-DD' format. strftime() works on this format directly.

**Quarter with CASE WHEN:**
```sql
CASE
    WHEN CAST(strftime('%m', ship_date) AS INTEGER) BETWEEN 1 AND 3  THEN 'Q1'
    WHEN CAST(strftime('%m', ship_date) AS INTEGER) BETWEEN 4 AND 6  THEN 'Q2'
    WHEN CAST(strftime('%m', ship_date) AS INTEGER) BETWEEN 7 AND 9  THEN 'Q3'
    ELSE 'Q4'
END AS quarter
```

In [10]:
# EXAMPLE — extract year, month, and year-month from ship_date
q = """
SELECT ship_date,
       strftime('%Y', ship_date)    AS year,
       strftime('%m', ship_date)    AS month,
       strftime('%Y-%m', ship_date) AS year_month,
       CASE
           WHEN CAST(strftime('%m', ship_date) AS INTEGER) BETWEEN 1 AND 3  THEN 'Q1'
           WHEN CAST(strftime('%m', ship_date) AS INTEGER) BETWEEN 4 AND 6  THEN 'Q2'
           WHEN CAST(strftime('%m', ship_date) AS INTEGER) BETWEEN 7 AND 9  THEN 'Q3'
           ELSE 'Q4'
       END AS quarter
FROM shipments
LIMIT 10
"""
df = pd.read_sql_query(q, conn)
display(df)

,ship_date,year,month,year_month,quarter
0,2024-11-23,2024,11,2024-11,Q4
1,2024-12-12,2024,12,2024-12,Q4
2,2024-01-16,2024,01,2024-01,Q1
3,2024-11-28,2024,11,2024-11,Q4
4,2024-01-04,2024,01,2024-01,Q1
5,2024-04-20,2024,04,2024-04,Q2
6,2024-06-25,2024,06,2024-06,Q2
7,2024-03-04,2024,03,2024-03,Q1
8,2024-12-26,2024,12,2024-12,Q4
9,2024-07-13,2024,07,2024-07,Q3


---
## Drill 1 — Monthly delivered shipment count and total freight

Show total delivered shipments and total freight_charge per month.  
Columns: year_month, shipment_count, total_freight. Order by year_month ascending.

Hint: GROUP BY strftime('%Y-%m', ship_date)

In [11]:
q1 = """ SELECT     strftime('%Y-%m', ship_date) AS year_month,
                    COUNT(shipment_id) AS shipment_count,
                   SUM(freight_charge) AS total_freight

FROM shipments 
WHERE status = 'Delivered'
GROUP BY year_month
ORDER BY year_month ASC

"""
df = pd.read_sql_query(q1, conn)
display(df)

,year_month,shipment_count,total_freight
0,2024-01,8,23419.09
1,2024-02,5,7405.26
2,2024-03,5,12486.47
3,2024-04,8,17707.74
4,2024-05,10,23969.77
5,2024-06,10,19492.80
6,2024-07,6,14277.86
7,2024-08,12,27379.60
8,2024-09,11,23872.19
9,2024-10,12,27205.40


---
## Drill 2 — Quarterly freight total and shipment count

Show total freight and shipment count per quarter — delivered only.  
Columns: quarter, shipment_count, total_freight. Order by quarter.

Hint: use the CASE WHEN quarter pattern from the concept cell.

In [12]:
q2 = """ SELECT CASE
    WHEN CAST(strftime('%m', ship_date) AS INTEGER) BETWEEN 1 AND 3  THEN 'Q1'
    WHEN CAST(strftime('%m', ship_date) AS INTEGER) BETWEEN 4 AND 6  THEN 'Q2'
    WHEN CAST(strftime('%m', ship_date) AS INTEGER) BETWEEN 7 AND 9  THEN 'Q3'
    ELSE 'Q4'
END AS quarter,
    COUNT(shipment_id) AS shipment_count,
    SUM(freight_charge) AS total_freight
FROM shipments
WHERE status = 'Delivered'
GROUP BY quarter
ORDER BY quarter ASC

"""
df = pd.read_sql_query(q2, conn)
display(df)

,quarter,shipment_count,total_freight
0,Q1,18,43310.82
1,Q2,28,61170.31
2,Q3,29,65529.65
3,Q4,34,71601.25


---
## Drill 3 — Shipment count by day of month

Show how many delivered shipments occurred on each day of the month (1–31).  
Columns: day_of_month, shipment_count. Order by day_of_month ascending.

Hint: CAST(strftime('%d', ship_date) AS INTEGER) for proper numeric sorting.

In [13]:
q3 = """
SELECT CAST(strftime('%d', ship_date) AS INTEGER) AS day_of_month,
        COUNT(*) AS shipment_count
FROM shipments
WHERE status = 'Delivered'
GROUP BY day_of_month


"""
df = pd.read_sql_query(q3, conn)
display(df)

,day_of_month,shipment_count
0,1,1
1,2,6
2,3,4
3,4,5
4,5,5
5,6,2
6,7,6
7,8,4
8,9,3
9,10,3


---
## Concept 2 — Date Arithmetic: Calculating Differences

SQLite doesn't have a `DATEDIFF()` function. Instead, use `julianday()` which converts a date to a floating-point number of days since a reference point. Subtracting two julianday values gives the difference in days.

```sql
-- Days between ship_date and delivery_date
julianday(delivery_date) - julianday(ship_date) AS days_to_deliver

-- Cast to integer to remove decimal
CAST(julianday(delivery_date) - julianday(ship_date) AS INTEGER) AS days_to_deliver
```

**Other date arithmetic:**
```sql
-- Add days to a date
date(ship_date, '+7 days')     -- 7 days after ship_date

-- Subtract days
date(ship_date, '-30 days')    -- 30 days before ship_date

-- Days since a fixed reference
julianday('2024-12-31') - julianday(hire_date)  -- days since hire
```

**Important:** julianday() returns NULL if either date is NULL — always filter for non-NULL delivery_date when calculating delivery time.

In [14]:
# EXAMPLE — delivery time in days for each delivered shipment
q = """
SELECT shipment_id,
       ship_date,
       delivery_date,
       CAST(julianday(delivery_date) - julianday(ship_date) AS INTEGER) AS days_to_deliver
FROM shipments
WHERE status = 'Delivered'
  AND delivery_date IS NOT NULL
ORDER BY days_to_deliver DESC
LIMIT 10
"""
df = pd.read_sql_query(q, conn)
display(df)

,shipment_id,ship_date,delivery_date,days_to_deliver
0,1018,2024-12-14,2024-12-24,10
1,1023,2024-09-28,2024-10-08,10
2,1100,2024-08-26,2024-09-05,10
3,1117,2024-08-07,2024-08-17,10
4,1147,2024-10-04,2024-10-14,10
5,1174,2024-10-10,2024-10-20,10
6,1188,2024-10-11,2024-10-21,10
7,1030,2024-06-18,2024-06-27,9
8,1047,2024-10-02,2024-10-11,9
9,1054,2024-12-19,2024-12-28,9


---
## Drill 4 — Average delivery time per route

Show average days to deliver per route_id — delivered shipments with non-NULL delivery_date only.  
Columns: route_id, avg_days_to_deliver (rounded to 1dp), total_shipments. Order by avg_days_to_deliver descending.

Hint: ROUND(AVG(julianday(delivery_date) - julianday(ship_date)), 1)

In [15]:
q4 = """
SELECT route_id,
        COUNT(shipment_id) AS total_shipments,
        ROUND(AVG(julianday(delivery_date)- julianday(ship_date)),1) AS avg_days_to_deliver
FROM shipments
WHERE status = 'Delivered'
AND delivery_date IS NOT NULL
GROUP BY route_id
ORDER BY avg_days_to_deliver DESC 
"""
df = pd.read_sql_query(q4, conn)
display(df)

,route_id,total_shipments,avg_days_to_deliver
0,2,13,6.6
1,3,8,6.5
2,6,10,6.4
3,1,8,5.8
4,4,10,5.3
5,10,13,5.2
6,5,9,5.1
7,9,7,5.0
8,7,8,5.0
9,12,6,4.7


---
## Drill 5 — Shipments that took more than 7 days to deliver

Show shipments where delivery took longer than 7 days.  
Columns: shipment_id, driver_id, ship_date, delivery_date, days_to_deliver. Longest first.

In [16]:
q5 = """ SELECT shipment_id,
                driver_id,
                ship_date,
                delivery_date,
                CAST(julianday(delivery_date)-julianday(ship_date) AS INTEGER) AS days_to_deliver
    FROM shipments
    WHERE status = 'Delivered'
    AND ship_date IS NOT NULL AND delivery_date IS NOT NULL
    AND days_to_deliver > 7
    ORDER BY days_to_deliver DESC
"""
df = pd.read_sql_query(q5, conn)
display(df)

,shipment_id,driver_id,ship_date,delivery_date,days_to_deliver
0,1018,8,2024-12-14,2024-12-24,10
1,1023,6,2024-09-28,2024-10-08,10
2,1100,5,2024-08-26,2024-09-05,10
3,1117,2,2024-08-07,2024-08-17,10
4,1147,2,2024-10-04,2024-10-14,10
5,1174,3,2024-10-10,2024-10-20,10
6,1188,2,2024-10-11,2024-10-21,10
7,1030,9,2024-06-18,2024-06-27,9
8,1047,4,2024-10-02,2024-10-11,9
9,1054,9,2024-12-19,2024-12-28,9


---
## Drill 6 — Average delivery time per driver

Join to drivers for name. Show average delivery time per driver — delivered only.  
Columns: name, province, avg_days (rounded 1dp), shipment_count. Order by avg_days descending.

In [17]:
q6 = """
SELECT d.name,
        Round(AVG(julianday(delivery_date)-julianday(ship_date)),1) AS avg_days,
        COUNT(shipment_id) AS shipment_count
FROM shipments s JOIN drivers d ON s.driver_id = d.driver_id
WHERE s.status = 'Delivered'
GROUP BY d.name 
ORDER BY avg_days DESC
"""
df = pd.read_sql_query(q6, conn)
display(df)

,name,avg_days,shipment_count
0,Mike Okafor,6.2,13
1,James Leblanc,6.1,13
2,Aisha Diallo,6.1,15
3,Priya Mehta,5.7,9
4,Sara Singh,5.5,15
5,Carlos Rivera,5.5,12
6,Linda Park,4.8,12
7,Ryan Kowalski,4.5,8
8,Jean Tremblay,4.3,3
9,Tom Beaumont,3.9,9


---
## Concept 3 — Filtering on Dates

Because SQLite stores dates as 'YYYY-MM-DD' text, string comparison works correctly for date filtering — alphabetical order matches chronological order for this format.

```sql
-- Filter a specific month
WHERE strftime('%Y-%m', ship_date) = '2024-03'

-- Filter a date range
WHERE ship_date >= '2024-10-01' AND ship_date <= '2024-12-31'

-- Filter a specific year
WHERE strftime('%Y', ship_date) = '2024'

-- Filter last N days relative to a reference date
WHERE ship_date >= date('2024-12-31', '-90 days')
```

**Rule:** Always filter on the raw date column in WHERE — avoid wrapping it in a function if possible, as this can prevent index use in production databases. `WHERE ship_date >= '2024-10-01'` is better than `WHERE strftime('%Y', ship_date) = '2024'` when filtering a full year.

In [18]:
# EXAMPLE — Q4 2024 delivered shipments (October–December)
q = """
SELECT shipment_id,
       ship_date,
       freight_charge,
       strftime('%Y-%m', ship_date) AS year_month
FROM shipments
WHERE status = 'Delivered'
  AND ship_date >= '2024-10-01'
  AND ship_date <= '2024-12-31'
ORDER BY ship_date
LIMIT 10
"""
df = pd.read_sql_query(q, conn)
display(df)

,shipment_id,ship_date,freight_charge,year_month
0,1047,2024-10-02,1303.62,2024-10
1,1168,2024-10-02,2123.35,2024-10
2,1147,2024-10-04,447.63,2024-10
3,1135,2024-10-09,1924.76,2024-10
4,1067,2024-10-10,1978.94,2024-10
5,1174,2024-10-10,1089.39,2024-10
6,1188,2024-10-11,4311.32,2024-10
7,1068,2024-10-18,2533.53,2024-10
8,1192,2024-10-20,3826.42,2024-10
9,1127,2024-10-22,2196.68,2024-10


---
## Drill 7 — Q1 2024 vs Q3 2024 freight comparison

Show total delivered freight for Q1 (Jan–Mar) and Q3 (Jul–Sep) side by side.  
Columns: quarter, total_freight, shipment_count.

Hint: filter using date ranges in WHERE and UNION, or use CASE WHEN quarter grouping.

In [19]:
q7 = """ SELECT CASE 
           WHEN CAST(strftime('%m', ship_date) AS INTEGER) BETWEEN 1 AND 3  THEN 'Q1'
           WHEN CAST(strftime('%m', ship_date) AS INTEGER) BETWEEN 7 AND 9  THEN 'Q3'
           END AS quarter,
           SUM(freight_charge) AS total_freight,
           COUNT(shipment_id) AS shipment_count
FROM shipments
WHERE status = 'Delivered'
GROUP BY quarter
HAVING quarter IS NOT NULL

"""
df = pd.read_sql_query(q7, conn)
display(df)

,quarter,total_freight,shipment_count
0,Q1,43310.82,18
1,Q3,65529.65,29


---
## Drill 8 — Shipments in the last 90 days of the dataset

The dataset ends on 2024-12-30. Show all shipments from the last 90 days of the dataset (on or after 2024-10-01).  
Columns: shipment_id, ship_date, status, freight_charge. Order by ship_date descending.

In [20]:
q8 = """ SELECT shipment_id,
                ship_date,
                status,
                freight_charge
FROM shipments 
WHERE ship_date >= date('2024-12-31', '-90 days') 
ORDER BY ship_date DESC
"""
df = pd.read_sql_query(q8, conn)
display(df)

,shipment_id,ship_date,status,freight_charge
0,1183,2024-12-30,Delivered,4476.63
1,1009,2024-12-26,Delivered,3026.50
2,1194,2024-12-25,Cancelled,1494.08
3,1112,2024-12-23,Delivered,596.26
4,1074,2024-12-22,Delivered,1888.49
5,1054,2024-12-19,Delivered,306.72
6,1109,2024-12-19,Delivered,1091.27
7,1181,2024-12-17,Delivered,3248.95
8,1018,2024-12-14,Delivered,426.35
9,1002,2024-12-12,Delayed,4030.98


---
## Drill 9 — Month-over-month freight change using LAG()

CTE `monthly`: total delivered freight per year_month.  
Main query: add LAG(total_freight) OVER (ORDER BY year_month) as prev_month_freight and calculate the change.  
Columns: year_month, total_freight, prev_month_freight, change. Order by year_month.

In [21]:
q9 = """
WITH monthly AS (   SELECT strftime('%Y-%m', delivery_date) AS year_month,
                            SUM(freight_charge) AS total_freight
                    FROM shipments
                    WHERE status = 'Delivered'
                    GROUP BY year_month),
with_lag AS (
    SELECT year_month,
           total_freight,
           LAG(total_freight) OVER (ORDER BY year_month) AS prev_month_freight
    FROM monthly)

SELECT year_month,
        total_freight,
        prev_month_freight,
        printf('%+.1f%%', (total_freight / prev_month_freight) * 100 - 100) AS change
FROM with_lag
ORDER BY year_month

"""
df = pd.read_sql_query(q9, conn)
display(df)

,year_month,total_freight,prev_month_freight,change
0,2024-01,23419.09,NaN,+0.0%
1,2024-02,3376.72,23419.09,-85.6%
2,2024-03,16515.01,3376.72,+389.1%
3,2024-04,12350.61,16515.01,-25.2%
4,2024-05,24636.08,12350.61,+99.5%
5,2024-06,19201.40,24636.08,-22.1%
6,2024-07,12385.60,19201.40,-35.5%
7,2024-08,32559.99,12385.60,+162.9%
8,2024-09,14819.85,32559.99,-54.5%
9,2024-10,35136.79,14819.85,+137.1%


---
## Concept 4 — Driver Tenure and Time-Based Segmentation

You can calculate how long a driver has been employed, segment drivers by tenure, and compare performance across cohorts.

```sql
-- Days employed (as of end of 2024)
CAST(julianday('2024-12-31') - julianday(hire_date) AS INTEGER) AS days_employed

-- Years employed (approximate)
ROUND((julianday('2024-12-31') - julianday(hire_date)) / 365.25, 1) AS years_employed

-- Tenure bucket
CASE
    WHEN julianday('2024-12-31') - julianday(hire_date) < 365  THEN 'Junior (<1yr)'
    WHEN julianday('2024-12-31') - julianday(hire_date) < 1095 THEN 'Mid (1-3yr)'
    ELSE 'Senior (3yr+)'
END AS tenure_band
```

In [22]:
# EXAMPLE — driver tenure as of end of 2024
q = """
SELECT name,
       province,
       hire_date,
       ROUND((julianday('2024-12-31') - julianday(hire_date)) / 365.25, 1) AS years_employed,
       CASE
           WHEN julianday('2024-12-31') - julianday(hire_date) < 365  THEN 'Junior (<1yr)'
           WHEN julianday('2024-12-31') - julianday(hire_date) < 1095 THEN 'Mid (1-3yr)'
           ELSE 'Senior (3yr+)'
       END AS tenure_band
FROM drivers
WHERE status = 'Active'
ORDER BY years_employed DESC
"""
df = pd.read_sql_query(q, conn)
display(df)

,name,province,hire_date,years_employed,tenure_band
0,Sara Singh,BC,2018-11-20,6.1,Senior (3yr+)
1,Linda Park,BC,2019-01-08,6.0,Senior (3yr+)
2,Jean Tremblay,QC,2019-03-15,5.8,Senior (3yr+)
3,Mike Okafor,ON,2020-07-01,4.5,Senior (3yr+)
4,Carlos Rivera,MB,2020-09-14,4.3,Senior (3yr+)
5,Tom Beaumont,AB,2021-02-10,3.9,Senior (3yr+)
6,Priya Mehta,ON,2022-05-30,2.6,Mid (1-3yr)
7,Ryan Kowalski,ON,2022-08-22,2.4,Mid (1-3yr)
8,James Leblanc,NS,2023-03-01,1.8,Mid (1-3yr)


---
## Drill 10 — Average freight per shipment by driver tenure band

CTE `driver_tenure`: add tenure_band to each active driver using the CASE WHEN above.  
CTE `driver_performance`: total delivered freight and shipment count per driver (join to shipments).  
Main query: join both CTEs, group by tenure_band, show avg freight per shipment.  
Columns: tenure_band, driver_count, total_shipments, avg_freight_per_shipment (rounded 2dp).

In [23]:
q10 = """

WITH driver_tenure AS ( SELECT d1.driver_id,
                     CASE
           WHEN julianday('2024-12-31') - julianday(hire_date) < 365  THEN 'Junior (<1yr)'
           WHEN julianday('2024-12-31') - julianday(hire_date) < 1095 THEN 'Mid (1-3yr)'
           ELSE 'Senior (3yr+)'
             END AS tenure_band 
                        FROM drivers d1),
    driver_performance AS ( SELECT d.driver_id,
                                    SUM(freight_charge) AS total_freight,
                                    COUNT(shipment_id) AS shipment_count
                            FROM shipments s JOIN drivers d ON s.driver_id = d.driver_id
                            WHERE s.status = 'Delivered'
                            GROUP BY d.driver_id)

    SELECT  tenure_band,
            COUNT(DISTINCT dp.driver_id) AS driver_count,
            SUM(dp.shipment_count) AS total_shipments,
           ROUND(AVG(total_freight),2) AS avg_freight_per_shipment
    FROM driver_performance dp JOIN driver_tenure dt ON dp.driver_id = dt.driver_id
    GROUP BY tenure_band

"""
df = pd.read_sql_query(q10, conn)
display(df)

,tenure_band,driver_count,total_shipments,avg_freight_per_shipment
0,Mid (1-3yr),3,30,16621.02
1,Senior (3yr+),7,79,27392.71


---
## Drill 11 — Drivers hired in each year — how many and total freight earned

Group drivers by their hire year. Show total delivered freight and shipment count for drivers hired each year.  
Columns: hire_year, driver_count, total_shipments, total_freight. Order by hire_year.

In [24]:
q11 = """
SELECT strftime('%Y', d.hire_date) AS hire_year,
        COUNT(DISTINCT d.driver_id) AS driver_count,
        COUNT(s.shipment_id) AS shipment_count,
        SUM(s.freight_charge) AS total_freight
        
FROM shipments s JOIN drivers d ON s.driver_id = d.driver_id
WHERE S.status = 'Delivered'
GROUP BY hire_year
ORDER BY hire_year 

"""
df = pd.read_sql_query(q11, conn)
display(df)

,hire_year,driver_count,shipment_count,total_freight
0,2018,1,15,37379.58
1,2019,2,15,39770.18
2,2020,2,25,57597.11
3,2021,2,24,57002.10
4,2022,2,17,22037.77
5,2023,1,13,27825.29


---
## Drill 12 — Proof drill: NULL delivery dates

Run two queries:
- Query A: average delivery days WITHOUT filtering `delivery_date IS NOT NULL`
- Query B: average delivery days WITH the filter

Do they differ? What does this tell you about how SQLite handles NULLs in AVG()?

In [25]:
# Query A — no NULL filter
q12a = """  SELECT AVG(julianday(delivery_date)-julianday(ship_date)) AS average_delivery_days
FROM shipments
WHERE status = 'Delivered'

"""
df = pd.read_sql_query(q12a, conn)
display(df)

,average_delivery_days
0,5.431193


In [26]:
# Query B — with NULL filter
q12b = """
SELECT AVG(julianday(delivery_date)-julianday(ship_date)) AS average_delivery_days
FROM shipments
WHERE delivery_date IS NOT NULL
"""
df = pd.read_sql_query(q12b, conn)
display(df)

,average_delivery_days
0,5.431193


**What did you observe? Does AVG() include or ignore NULLs?**

> The AVG() function isnt affected as delivered_date already depends on a delivered shipment inherently. Either filter in this case is redundant.

---
## Block 2 · 11:00am — Applied Challenge

> *The operations manager's full request: month-by-month freight trend, strongest quarter, average delivery time, and whether newer drivers outperform veterans.*

**Query 1:** Monthly trend — total freight per month with month-over-month change and running total for 2024  
**Query 2:** Quarterly performance — which quarter had the highest freight, highest count, and best avg delivery time  
**Query 3:** Tenure vs performance — do Senior, Mid, or Junior drivers have better avg freight per shipment AND faster delivery times?

Write your framing sentence first.

**My approach — what date functions and CTEs are needed for each query:**

> For query 1 I need: a cte to calculate total freight per month using a date function to isolate each month of the year. Then a window function CTE to calculate month over month change. Then another CTE calculating running total using SUM() function of the total freight calculated in the first CTE.

In [27]:
# Query 1 — Monthly trend with MoM change and running total
q13 = """
        WITH month_freight AS (    SELECT strftime('%Y-%m', delivery_date) AS month,
                                            SUM(freight_charge) AS total_freight
                                    FROM shipments 
                                    WHERE status = 'Delivered'
                                    GROUP BY month),
        monthly_running_freight AS (  SELECT month,
                                total_freight,
                                SUM(total_freight) OVER (ORDER BY month) AS running_freight
                                FROM month_freight),
        change AS (   SELECT  month,
                        total_freight,
                        running_freight,
                        LAG(total_freight) OVER (ORDER BY month) AS prev_freight
                        FROM monthly_running_freight
                        
                        )
        SELECT month,
                total_freight,
                 running_freight,
                 ROUND((total_freight - prev_freight)/ prev_freight *100, 1) AS pct_change
        FROM change
"""
df = pd.read_sql_query(q13, conn)
display(df)

,month,total_freight,running_freight,pct_change
0,2024-01,23419.09,23419.09,NaN
1,2024-02,3376.72,26795.81,-85.6
2,2024-03,16515.01,43310.82,389.1
3,2024-04,12350.61,55661.43,-25.2
4,2024-05,24636.08,80297.51,99.5
5,2024-06,19201.40,99498.91,-22.1
6,2024-07,12385.60,111884.51,-35.5
7,2024-08,32559.99,144444.50,162.9
8,2024-09,14819.85,159264.35,-54.5
9,2024-10,35136.79,194401.14,137.1


In [31]:
# Query 2 — Quarterly performance Quarterly performance — which quarter had the highest freight, highest count, and best avg delivery time 
q14 = """

WITH quarter_performance AS (SELECT CASE 
                                    WHEN CAST(strftime('%m', ship_date) AS INTEGER) BETWEEN 1 AND 3 THEN 'Q1'
                                    WHEN CAST(strftime('%m', ship_date) AS INTEGER) BETWEEN 4 AND 6 THEN 'Q2'
                                    WHEN CAST(strftime('%m', ship_date) AS INTEGER) BETWEEN 7 AND 9 THEN 'Q3'
                                     ELSE 'Q4'END AS quarter,   
                                      SUM(freight_charge) AS total_freight,
                                    COUNT(shipment_id) AS shipment_count,
                                     ROUND(AVG(julianday(delivery_date) - julianday(ship_date)), 2) AS avg_delivery_time
                                    FROM shipments 
                                    WHERE status = 'Delivered'
                                     GROUP BY quarter),
    
     ranked AS ( SELECT *,
        DENSE_RANK() OVER (ORDER BY total_freight DESC) AS freight_rank,
        DENSE_RANK() OVER (ORDER BY shipment_count DESC) AS count_rank,
        DENSE_RANK() OVER (ORDER BY avg_delivery_time ASC) AS delivery_rank
    FROM quarter_performance)
   
SELECT 
    quarter,
    total_freight,
    shipment_count,
    avg_delivery_time
FROM ranked
WHERE freight_rank = 1 OR count_rank = 1 OR delivery_rank = 1

   
        
"""
df = pd.read_sql_query(q14, conn)
display(df)

,quarter,total_freight,shipment_count,avg_delivery_time
0,Q4,71601.25,34,6.24
1,Q2,61170.31,28,4.86


In [ ]:
q15 = """
WITH tenure AS (
    SELECT  driver_id,
            CASE
                WHEN julianday('2024-12-31') - julianday(hire_date) < 365  THEN 'Junior (<1yr)'
                WHEN julianday('2024-12-31') - julianday(hire_date) < 1095 THEN 'Mid (1-3yr)'
                ELSE 'Senior (3yr+)'
            END AS tenure_band
    FROM drivers
),
performance AS (
    SELECT  t.tenure_band,
            COUNT(s.shipment_id) AS shipment_count,
            SUM(s.freight_charge) AS total_freight,
            ROUND(AVG(julianday(s.delivery_date) - julianday(s.ship_date)), 2) AS avg_delivery_time
    FROM shipments s JOIN tenure t ON s.driver_id = t.driver_id
    WHERE s.status = 'Delivered'
    GROUP BY t.tenure_band
),
ranked AS (
    SELECT  tenure_band,
            shipment_count,
            total_freight,
            avg_delivery_time,
            DENSE_RANK() OVER (ORDER BY total_freight DESC)    AS freight_rank,
            DENSE_RANK() OVER (ORDER BY shipment_count DESC)   AS count_rank,
            DENSE_RANK() OVER (ORDER BY avg_delivery_time ASC) AS delivery_rank
    FROM performance
)

SELECT
    (SELECT tenure_band FROM ranked WHERE freight_rank = 1)   AS highest_freight,
    (SELECT tenure_band FROM ranked WHERE count_rank = 1)     AS most_orders,
    (SELECT tenure_band FROM ranked WHERE delivery_rank = 1)  AS quickest_delivery
"""

df = pd.read_sql_query(q15, conn)
display(df)

,highest_freight,most_orders,quickest_delivery
0,Senior (3yr+),Senior (3yr+),Senior (3yr+)


**What does the data say? What would you tell the operations manager?**

> *(write here)*

---
## Capstone — Time-Based Operations Report

Build a single query using CTEs that produces a full time-based performance report by quarter and driver tenure band:

| Column | Definition |
|---|---|
| quarter | Q1 / Q2 / Q3 / Q4 |
| tenure_band | Junior / Mid / Senior |
| shipment_count | Delivered shipments in that quarter by that tenure band |
| total_freight | Total freight_charge |
| avg_freight | Average freight per shipment (rounded 2dp) |
| avg_delivery_days | Average delivery time in days (rounded 1dp) |
| pct_of_quarter_freight | This tenure band's freight as % of that quarter's total (rounded 1dp) |

Order by quarter, then total_freight descending.

Hint: suggested CTEs — `with_dates` (add quarter + tenure_band), `quarter_totals` (total per quarter for % calc), final SELECT joins both.

In [ ]:
q_capstone = """

"""
df = pd.read_sql_query(q_capstone, conn)
display(df)

---
## Day 3 Checkpoint — answer before Day 4

Plain English — no SQL needed.

**1. What function does SQLite use to extract date parts, and how would you get the year-month from a date column?**

> SQLite uses the strftime() function to extract or format parts of a date. To get the year and month from a date column, you would use: strftime('%Y-%m', date_column)

**2. SQLite has no DATEDIFF(). How do you calculate the number of days between two dates?**

> SQLite does not have a DATEDIFF() function, so we use JULIANDAY() to convert date values into a numeric representation based on Julian days. We can then subtract the start date from the end date to calculate the difference in days: JULIANDAY(end_date) - JULIANDAY(start_date)

**3. Why is `WHERE ship_date >= '2024-10-01'` generally better than `WHERE strftime('%m', ship_date) >= '10'` for date filtering?**

> WHERE ship_date >= '2024-10-01' is better because it compares the raw date values directly, which allows SQLite to use indexes on the ship_date column and perform an efficient range filter. In contrast, WHERE strftime('%m', ship_date) >= '10' applies a function to every row, which prevents index usage.

**4. The operations manager asks "are newer drivers faster at delivering?" What columns and calculations do you need, and what's your plan before writing a single line of SQL?**

> I would first identify the key columns needed: driver ID, hire date, shipment ID, pickup time, and delivery time. From these, I would calculate delivery duration per shipment and then aggregate it to get average delivery time per driver. Next, I would define “newer” and “older” drivers based on hire date or tenure and compare their average delivery performance. Finally, I would analyze whether there is a statistically meaningful difference between the two groups. Before writing SQL, my plan is to break the problem into three steps: compute shipment delivery times, aggregate performance by driver, and then segment drivers by tenure to compare results.